<a href="https://colab.research.google.com/github/MuhammadQasimTanveer/HerdOrbit-ML/blob/main/image_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device:", device)

In [ ]:
dataset_path = Path("/content/drive/MyDrive/HerdOrbit_Disease")

for folder in dataset_path.iterdir():
    if folder.is_dir():
        print(folder.name)

In [ ]:
for folder in dataset_path.iterdir():
    if folder.is_dir():
        count = len(list(folder.iterdir()))
        print(folder.name, ":", count)

In [ ]:
from pathlib import Path
split_path = Path("/content/drive/MyDrive/HerdOrbit_Disease_Split")
print(split_path)

In [ ]:
import random
import shutil

random.seed(42)
split_path = Path("/content/drive/MyDrive/HerdOrbit_Disease_Split")

for folder in dataset_path.iterdir():
    if folder.is_dir():

        images = list(folder.iterdir())
        random.shuffle(images)

        train = int(len(images) * 0.8)
        val = int(len(images) * 0.9)

        for name, files in { "train": images[:train],"val": images[train:val],"test": images[val:]}.items():
            new_folder = split_path / name / folder.name
            new_folder.mkdir(parents=True, exist_ok=True)

            for image in files:
                shutil.copy(image, new_folder / image.name)

print("Dataset split completed.")

In [ ]:
for split in ["train", "val", "test"]:
    print("\n", split.upper())

    for folder in (split_path / split).iterdir():
        if folder.is_dir():
            print(folder.name, ":", len(list(folder.iterdir())))

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Preprocessing and augmentation ready.")

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter
import torch
import numpy as np

# Load datasets
train_dataset = datasets.ImageFolder(split_path / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(split_path / "val", transform=val_test_transform)
test_dataset = datasets.ImageFolder(split_path / "test", transform=val_test_transform)

# Get training labels and class counts
targets = [label for _, label in train_dataset.samples]
class_counts = Counter(targets)
num_classes = len(train_dataset.classes)

# Give higher sampling probability to smaller classes
sample_weights = [1.0 / class_counts[label] for label in targets]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# Calculate moderate class weights
counts = np.array([class_counts[i] for i in range(num_classes)])
raw_weights = 1.0 / counts
class_weights = raw_weights / raw_weights.sum() * num_classes

# Reduce the strength of the weights
class_weights = np.sqrt(class_weights)

class_weights = torch.tensor(class_weights, dtype=torch.float)

# Create DataLoaders
train_loader = DataLoader(train_dataset,batch_size=32,sampler=sampler)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

print("Classes:", train_dataset.classes)
print("Class counts:", dict(class_counts))
print("Class weights:", class_weights)

In [ ]:
# Load pretrained EfficientNet-B0
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

# Freeze pretrained layers
for p in model.parameters():
    p.requires_grad = False

# Replace classifier for 6 disease classes
num_classes = len(train_dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Enable classifier training
for p in model.classifier.parameters():
    p.requires_grad = True

# Move model to GPU
model = model.to(device)
print("EfficientNet-B0 ready.")
print("Classes:", num_classes)

In [ ]:
import torch.optim as optim

# Freeze the pretrained backbone, Train only the classifier
for p in model.parameters():
    p.requires_grad = False

for p in model.classifier.parameters():
    p.requires_grad = True

# Use class weights for imbalanced classes
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimize classifier parameters
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print("Classifier training ready.")